# 01 - Data Cleaning and Implied-Volatility Surface

**Objective:** Build the cleaned AAPL option surface used by the rest of the project.

**Inputs:** Live `yfinance` option chains, or an existing saved CSV under `data/processed/`.

**Outputs:** `data/processed/AAPL_surface_checked.csv`, `data/processed/AAPL_iv_fit.csv`, and market/surface figures under `reports/figures/`.

**Key takeaway:** The notebook turns noisy option-chain quotes into a validated volatility representation suitable for calibration.



In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "heston_matplotlib_cache"))

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

from src.data import build_option_dataset, load_option_dataset
from src.validation import validate_dataset, infer_rate_dividend_by_expiry, attach_rates_and_recheck
from src.plotting import fit_iv_surface, plot_iv_surface

## Configuration

The default ticker is AAPL because its option chain is liquid enough for a clean volatility-smile demonstration. Set `REFRESH_DATA = True` to pull fresh data with `yfinance`; set it to `False` to reuse the saved CSV.

In [ ]:
TICKER = "AAPL"
REFRESH_DATA = True

processed_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface.csv"
checked_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface_checked.csv"
iv_fit_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_iv_fit.csv"

In [ ]:
if REFRESH_DATA or not processed_path.exists():
    dataset, spot = build_option_dataset(
        ticker=TICKER,
        expiries="auto",
        tmin=0.03,
        tmax=0.60,
        moneyness_bounds=(0.70, 1.30),
        iv_bounds=(0.01, 2.50),
        require_open_interest=True,
        save_csv=processed_path,
    )
else:
    dataset = load_option_dataset(processed_path)
    spot = float((dataset["strike"] / dataset["moneyness"]).median())

print(f"Spot estimate: {spot:.4f}")
print(f"Rows: {len(dataset):,}; expiries: {dataset['expiry'].nunique()}")
dataset.head()

## Quote Validation

The first check reprices each option from its quoted implied volatility and compares that Black-Scholes price with the cleaned target price. The second check uses put-call parity:

$$C - P = S e^{-qT} - K e^{-rT}$$

Large residuals usually indicate stale, wide, or internally inconsistent quotes.

In [ ]:
checked, parity_pairs, summary = validate_dataset(dataset, spot=spot, rate=0.0, dividend=0.0)
summary

In [ ]:
rate_dividend_map = infer_rate_dividend_by_expiry(dataset, spot=spot)
checked_rq, parity_pairs_rq, summary_rq = attach_rates_and_recheck(dataset, spot, rate_dividend_map)

checked_rq.to_csv(checked_path, index=False)
summary_rq

## Surface Fit

We smooth implied volatility by expiry as a function of moneyness, then combine those smiles into a surface over maturity and moneyness. This fitted surface becomes the target object for Heston calibration.

In [ ]:
iv_fit_df = fit_iv_surface(checked_rq, outdir=FIG_DIR, smoothing=0.01)
plot_iv_surface(iv_fit_df, outdir=FIG_DIR)
iv_fit_df.to_csv(iv_fit_path, index=False)

print(f"Saved checked data to {checked_path.relative_to(PROJECT_ROOT)}")
print(f"Saved fitted surface to {iv_fit_path.relative_to(PROJECT_ROOT)}")
iv_fit_df.head()

## Takeaway

The cleaned option chain and fitted IV surface define the empirical object that the Black-Scholes diagnostics and Heston calibration use downstream.

